# Guide to the notebook

## Interface

- There is a console on the left to create cells temporarily for testing purpose. You can toggle it by click the **Console** button on the top toolbar.
- Click a cell and press **Ctrl-Enter** or **Cmd-Ender** will exectute the commands in cell
- If you get an unexpected result try to rerun the cell

# Practical - Pathogen genomics

Tuberculosis (TB) is an infectious disease caused by the bacterium Mycobacterium tuberculosis. According to the WHO, in 2018 there were 10.0 million new cases of TB worldwide and 1.4 million deaths due to the disease, making TB the world’s most deadly infectious disease. The publication of the genome of M. tuberculosis H37Rv in 1998 gave researchers a powerful new tool in understanding this pathogen. This genome has been revised since then, with the latest version being available as RefSeq entry NC_000962.3. The genome comprises a single circular chromosome of some 4.4 megabases. The H37Rv strain that the genome was sequenced from is a long-preserved laboratory strain, originally isolated from a patient in 1905 and named as H37Rv in 1935. It is notably different in some genomic regions from some modern clinical strains but remains the standard reference sequence for M. tuberculosis (Mtb). In a larger context M. tuberculosis is a prominent member of the Mycobacterium Tuberculosis Complex (MTBC).

This group of related species comprises of the 8 lineages of human-infecting M. tuberculosis as well as predominantly animal-infecting species such as M. bovis and M. pinnipedii. Two other close relatives of Mtb, M. leprae and M. lepromatosis circulate between humans, causing the disease leprosy. Finally amongst the Mycobacteria there are several other species that live in the environment and can cause human disease. These are the Nontuberculous Mycobacteria.

Variation in the genome of M. tuberculosis (Mtb) is associated with changes in phenotype, for example drug resistance and virulence. It is also useful for outbreak investigation as the single nucleotide polymorphisms (SNPs) in a sample can be used to build a phylogeny.

This tutorial will focus on identifying genomic variation in Mtb and using that to explore drug resistance and other aspects of the bacteria.


## Exercise A: Genomic Sequencing Data Analysis

## Preparation - Create your output folder

In [2]:
%%bash
# create the output folder for storing output files
if [ ! -d ./output ]; then
  mkdir ./output
fi

### 1. Quality control
This step serves the purpose of identifying possible issues with the raw sequenced reads input data before embarking on any “real” analysis steps.

Some of the typical problems with NGS data can be mitigated by preprocessing affected sequencing reads before trying to map them to the reference genome. Detecting some other, more severe problems early on may at least save you a lot of time spent on analyzing low-quality data that is not worth the effort.


In [ ]:
#Running FastQC
#for about 1 min
source /home/student/USER/fastqc.sh
mkdir -p ./output/1QC
fastqc -o ./output/1QC -t 8 /home/student/pathogen_genomics/data/*_1.fastq.gz /home/student/pathogen_genomics/data/*_2.fastq.gz

### Viewing QC Results
You can now right-click on the HTML files output in the /output/1QC/ folder and select "Open in new browser tab" to view the quality control (QC) reports.

## 2. Find variants
We have an annotated reference built from the inferred M. tuberculosis ancestral reference genome and the gene annotation from the H37Rv strain so will use it in this case.

In [ ]:
#see the pipeline
cat /home/student/pathogen_genomics/data/0pipeline/snp.sh

In [ ]:
#run the pipeline
#for about 20 min
source /home/student/USER/miniconda.sh
conda activate zhuxi
bash /home/student/pathogen_genomics/data/0pipeline/snp.sh
conda deactivate
conda deactivate

Your /output/2vcf/ directory should now contain the .vcf and .cns files generated during the pipeline execution.

### 3. Further variant filtering
We still cannot entirely trust the proposed variants. In particular, there are regions of the M. tuberculosis genome that are difficult to effectively map reads to. These include the PE/PPE/PGRS genes, which are highly repetitive, and the IS (insertion sequence sites). Secondly, when an insertion or deletion (indel) occurs in our sample relative to the reference it can cause apparent, but false, single nucleotide variants to appear near the indel. Finally where few reads map to a region of the reference genome, either because of a sequence deletion or because of a high GC content in the genomic region, we cannot be confident about the quality of variant calling in the region. The next step can help filter out variants based on a variety of criteria, including those listed above.

In [ ]:
# Filter out reads with depth < 5 and frequency < 0.75
for i in ./output/2vcf/*.all.vcf; do
    filename=$(basename "$i")                      
    sample=${filename%.all.vcf}                    
    perl /home/student/pathogen_genomics/data/0pipeline/filter_af_v2.pl "$i" > ./output/2vcf/${sample}.d5f75.vcf
done

# Filter out SNPs located in the pe/ppe genes
for i in ./output/2vcf/*.d5f75.vcf; do
    filename=$(basename "$i")                     
    sample=${filename%.d5f75.vcf}                  
    perl /home/student/pathogen_genomics/data/0pipeline/snpFilt_PPE.pl \
        /home/student/pathogen_genomics/data/0pipeline/Excluded_loci.txt "$i" > ./output/2vcf/${sample}.ppe.vcf
done


### Drug resistance prediction

In [ ]:
# Clear previous results
> ./output/2vcf/drug_mutation.txt

# Run drug resistance prediction and clean output paths
for file in ./output/2vcf/*.ppe.vcf; do
    name=$(basename "$file")  # e.g., CRR188797.ppe.vcf
    perl /home/student/pathogen_genomics/data/0pipeline/drug_predict.vcf.pl \
        /home/student/pathogen_genomics/data/0pipeline/drug_resistance_simutation_NokasA.txt "$file" | \
        sed "s|$file|$name|g" >> ./output/2vcf/drug_mutation.txt
done

# Display results
cat ./output/2vcf/drug_mutation.txt


In [ ]:
# Filter out drug resistance sites (they may affect phylogenetic reconstruction)
mkdir -p ./output/3fasta

for file in ./output/2vcf/*.ppe.vcf; do
    name=$(basename "$file")                    
    prefix=${name%.ppe.vcf}                      
    perl /home/student/pathogen_genomics/data/0pipeline/filter_drug_mutation.vcf.pl \
        /home/student/pathogen_genomics/data/0pipeline/drug_resistance_simutation_NokasA.txt "$file" \
        > ./output/3fasta/${prefix}.dr.ppe.vcf
done

In [ ]:
# Rename filtered VCF files by removing '.dr.ppe.vcf' suffix and replacing it with '.vcf'
for file in ./output/3fasta/*.dr.ppe.vcf; do
    mv "$file" "${file%dr.ppe.vcf}vcf"
done

# Convert filtered VCF files to SNP files
bash /home/student/pathogen_genomics/data/0pipeline/vcf_to_snp.sh

# Generate a list of all unique SNP positions across all strains
cut -f1 ./output/3fasta/*.snp | sort -n | uniq > ./output/3fasta/all_location.list

# Recall mutations using CNS and SNP files for each sample
for cns_path in ./output/2vcf/*.cns; do
    base_name=$(basename "$cns_path" .cns)
    snp_file="./output/3fasta/${base_name}.snp"
    out_fasta="./output/3fasta/${base_name}.fas"

    perl /home/student/pathogen_genomics/data/0pipeline/1_vsc2hpt_lc.pl \
        "$cns_path" \
        ./output/3fasta/all_location.list \
        "$snp_file" > "$out_fasta"
done

# Merge all individual .fas files into a single multi-strain FASTA file
cat ./output/3fasta/*.fas > ./output/3fasta/all_strains.fasta


In [ ]:
# Count the number of strains
strain_count=$(grep -c "^>" all_strains.fasta)
echo "Number of strains: $strain_count"

# Determine the alignment length (number of SNP sites)
alignment_length=$(awk '
    BEGIN { seqlen=0 }
    /^>/ { if (seqlen>0) { print seqlen; exit } }
    !/^>/ { seqlen += length($0) }
    END { if (seqlen>0) print seqlen }
' ./output/3fasta/all_strains.fasta)
echo "Alignment length per strain: $alignment_length"

In this exercise, we processed raw next-generation sequencing files (*.fastq.gz) and generated .fasta files containing mutation sites for all strains, including drug resistance prediction during the pipeline.


## Exercise B: Clustering Analysis and Phylogenetic Tree Building

Due to computational time constraints, for this section we will use a pre-prepared 'cluster.fasta' file containing 33 strain fasta files generated according to Exercise A for demonstration and practice.

### 1. Clustering analysis
In recent years, the rapid development of whole-genome sequencing technologies has brought tuberculosis transmission research into the era of genomic epidemiology. Single-nucleotide polymorphisms (SNPs) obtained from whole-genome sequencing provide the highest resolution for differentiating between strains while also allowing accurate construction of phylogenetic and evolutionary relationships among strains.

The most popular method for analyzing recent transmission is to group strains based on genetic distance. By performing pairwise comparisons of the whole-genome sequences of strains, if the SNP distance is less than or equal to a specific threshold (typically 12 SNPs), the strains are defined as clustered strains, which is usually interpreted as evidence of recent local transmission. If the SNP distance exceeds this threshold, the strains are defined as non-clustered strains. Patients with non-clustered (singleton) strains are considered newly imported cases (i.e., patients infected elsewhere).

On this basis, tracking and investigating tuberculosis patients with clustered strains enables the inference of transmission pathways and networks. This allows for quantification of the effects of individual, population, social, spatiotemporal factors, and pathogen genomic characteristics on tuberculosis transmission, helping to identify drivers across different dimensions and providing scientific guidance for tuberculosis control strategies.
#### 1.1 Pairwise SNP Distance Calculation Between Strains

In [ ]:
# Count the number of strains
strain_count=$(grep -c "^>" cluster.fasta)
echo "Number of strains: $strain_count"

# Determine the alignment length (number of SNP sites)
alignment_length=$(awk '
    BEGIN { seqlen=0 }
    /^>/ { if (seqlen>0) { print seqlen; exit } }
    !/^>/ { seqlen += length($0) }
    END { if (seqlen>0) print seqlen }
' /home/student/pathogen_genomics/data/4cluster/cluster.fasta)
echo "Alignment length per strain: $alignment_length"


In [ ]:
# View the strain list for clustering analysis
# The file 'cluster.list' contains the list of strain names included in the analysis
echo "Displaying strain list from cluster.list:"
cat /home/student/pathogen_genomics/data/4cluster/cluster.list

In [ ]:
mkdir -p ./output/4cluster
#Run the SNP distance calculation script
echo "Calculating pairwise SNP distances..."
python3 /home/student/pathogen_genomics/data/0pipeline/snp_distance.py /home/student/pathogen_genomics/data/4cluster/cluster.fasta /home/student/pathogen_genomics/data/4cluster/cluster.list ./output/4cluster/cluster_distance.txt

#View the resulting SNP distance matrix
echo "Displaying the calculated SNP distance matrix:"
cat ./output/4cluster/cluster_distance.txt

#### 1.2 Inference of Transmission Cluster
Use the Python networkx library to perform cluster network analysis, constructing the network structure between strains, counting the number of networks, and saving the analysis results as an csv file.
- Read and process SNP comparison data: Use pandas to read the file and extract strain pairs with SNP distances less than or equal to a given threshold (e.g., 12).

- Construct clustered networks: Use networkx to convert the clustered strain pairs into an undirected network and analyze its connected components.

- Assign network identifiers: Assign a corresponding network ID to each strain.

- Save results: Save the strains and their network IDs to a CSV file and output the total number of formed networks.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Step 1: Read SNP distance data and drop any NaN values
file_path = './output/4cluster/cluster_distance.txt'
data = pd.read_csv(file_path, sep='\t', names=['strain_a', 'strain_b', 'distance']).dropna()

# Step 2: Set SNP threshold and filter strain pairs below or equal to the threshold
threshold = 12
filtered_data = data[data['distance'] <= threshold]

# Step 3: Create an undirected graph from the filtered strain pairs
G = nx.from_pandas_edgelist(filtered_data, 'strain_a', 'strain_b')

# Step 4: Compute connected components and assign network IDs to each strain
network_data = [{'Strain': strain, 'Network_ID': i + 1}
                for i, component in enumerate(nx.connected_components(G))
                for strain in component]

# Step 5: Save results to a CSV file
network_df = pd.DataFrame(network_data)
output_file = './output/4cluster/network.csv'
network_df.to_csv(output_file, index=False)


# Step 6: Output summary to console
print(f'Total networks formed: {nx.number_connected_components(G)}')
print(f'Results have been saved to {output_file}')
print('Preview of the generated CSV:')
print(network_df)

Based on the 12 SNP threshold, these 33 strains have been divided into 4 genomic clusters.

Next, we will proceed with:

### 2. Phylogenetic Tree Building
Phylogeny refers to the formation and development process of a particular taxonomic group, in contrast to individual development, and it studies evolutionary relationships (origins and evolutionary history).
Phylogenetic analysis aims to infer (evaluate) these evolutionary relationships. The relationships inferred from phylogenetic analysis are typically represented as a phylogenetic tree, and the process of phylogenetic analysis includes constructing, evaluating, and interpreting phylogenetic trees.

Molecular Phylogenetic Tree Reconstruction Algorithms:
- Maximum Parsimony (MP)
- Distance Methods
- Maximum Likelihood (ML)
- Bayesian Inference (BI)

Computation speed (from fastest to slowest):
Distance > Maximum Parsimony > Maximum Likelihood > Bayesian Inference

Next, we will use iqtree(https://iqtree.github.io/) to construct a phylogenetic tree using the Maximum Likelihood (ML) method, which:

✅ Has a solid statistical foundation.
✅ Produces trees that best fit the evolutionary reality when the evolutionary model is appropriately selected.

In [ ]:
#Infer maximum-likelihood tree from a sequence alignment with the best-fit model automatically selected by ModelFinder:
source /home/student/USER/iqtree.sh
iqtree3 -s /home/student/pathogen_genomics/data/4cluster/cluster.fasta \
        -pre ./output/4cluster/cluster

In [ ]:
cat ./output/4cluster/cluster.iqtree

Next, we can use iTOL (Interactive Tree of Life, https://itol.embl.de/) for online visualization.

We can manually upload annotation files to customize the tree, including:

- Coloring different transmission clusters (4 genomic clusters).
- Annotating drug resistance mutation profiles on the tree.

Below is an example of a beautified tree exported from iTOL:

In [ ]:
from IPython.display import Image
Image("/home/student/pathogen_genomics/data/4cluster/cluster.jpg")

## Exercise C: Transmission Inference

### Simultaneous inference of phylogenetic and  transmission trees in infectious disease  outbreaks

It is becoming easier and cheaper to obtain whole-genome sequences of pathogen samples during infectious disease outbreaks. When all hosts involved in an outbreak are sampled and these samples are sequenced, the small differences between the sequences (single nucleotide polymorphisms, SNPs) provide information on the transmission tree—that is, who infected whom and when.

However, correctly inferring this transmission tree is not straightforward, as SNPs arise from unobserved processes, including infection events as well as pathogen growth and mutation within hosts. In recent years, several methods have been developed to address this problem, but these methods are often tailored for specific applications or make limiting assumptions, making them difficult to apply to new settings and datasets.

In this section, we introduce phybreak (doi: http://dx.doi.org/10.1101/069195), a method to infer transmission trees without imposing prior limiting constraints on the order of unobserved events. This method is easily accessible through an R package implementation and can be applied flexibly to new datasets and outbreak scenarios.
                                                                                                                                                                                                                                                                                                                                                                               
In this analysis, we use Cluster Network 2 from Exercise B as an example to perform transmission inference, aiming to infer who infected whom and to estimate the timing of transmission events.                                                                                                                                                                                                                                                                                                                                                                               

In [ ]:
mkdir -p ./output/5phybreak

for about 5 min

In [ ]:
# ----------------------------- #
# Load required libraries
# ----------------------------- #
library(phybreak)
library(ape)
library(coda)
library(openxlsx)
library(IRdisplay)   

# ----------------------------- #
# Step 1: Load data
# ----------------------------- #
cat("Reading input data...\n")
data <- read.table("/home/student/pathogen_genomics/data/5phybreak/c2_date.txt", header = TRUE, sep = "\t")
names <- data$name
dates <- as.numeric(as.character(data$date))
sequences <- read.FASTA('/home/student/pathogen_genomics/data/5phybreak/c2.fasta')

dataset <- phybreakdata(
  sequences = sequences,
  sample.times = dates
)

# ----------------------------- #
# Step 2: Initialize phybreak
# ----------------------------- #
cat("Initializing phybreak model...\n")
phybreak <- phybreak(
  dataset = dataset,
  gen.mean = 257, sample.mean = 189,
  mu = 1e-05,
  gen.shape = 4.66, sample.shape = 3.05,
  wh.model = 3, wh.slope = 1,
  est.gen.mean = TRUE, prior.mean.gen.mean = 1, prior.mean.gen.sd = Inf,
  est.sample.mean = TRUE, prior.mean.sample.mean = 1, prior.mean.sample.sd = Inf,
  est.wh.slope = TRUE, prior.wh.shape = 3, prior.wh.mean = 1,
  use.tree = FALSE
)

# ----------------------------- #
# Step 3: Burn-in and sampling
# ----------------------------- #
cat("Running burn-in (50,000 iterations)...\n")
phybreak <- burnin_phybreak(phybreak, ncycles = 5000)

cat("Running MCMC sampling (500,000 samples)...\n")
phybreak <- sample_phybreak(phybreak, nsample = 50000)

# ----------------------------- #
# Step 4: Calculate effective sample size
# ----------------------------- #
cat("Calculating effective sample sizes...\n")
mcmc <- get_mcmc(phybreak)  #
effective_size <- effectiveSize(mcmc)
effective_size_df <- data.frame(Parameter = names(effective_size), EffectiveSize = effective_size)
write.xlsx(effective_size_df, "./output/5phybreak/effective_c2.xlsx", rowNames = FALSE)

# ----------------------------- #
# Step 5: Extract transmission tree
# ----------------------------- #
cat("Extracting transmission tree...\n")
trans_tree <- transtree(phybreak, method = "edmonds")
trans_tree_df <- as.data.frame(trans_tree)
trans_tree_df$Patient <- rownames(trans_tree_df)
trans_tree_df <- trans_tree_df[, c("Patient", names(trans_tree_df)[-ncol(trans_tree_df)])]
write.xlsx(trans_tree_df, "./output/5phybreak/transtree_c2.xlsx", rowNames = TRUE)
IRdisplay::display(trans_tree_df)

# ----------------------------- #
# Step 6: Plot and show JPG
# ----------------------------- #
cat("Plotting and saving transmission tree as JPG...\n")
jpg_path <- "./output/5phybreak/c2.jpg"
jpeg(jpg_path, width = 1200, height = 1000, res = 150)
plotTrans(phybreak, plot.which = "edmonds")
dev.off()
IRdisplay::display_jpeg(file = jpg_path)


cat("Pipeline completed successfully.\n")
